## Создание обучающих данных из сырых датасетов

Опишу идею создания датасета. В AWS есть понятие SPOT-инстансов. Это инстансы, цена аренды за которые постоянно меняется. Такие инстансы предоставляются со значительной скидкой, но со скидкой мы получаем и риск выключения нашего инстанса с потерей всех данных. Несмотря на риски, их выгодно использовать для каких-либо кратковременных джоб. В данном проекте будет описываться предсказание "выживания" самого популярного SPOT-инстанса с GPU (g5.2xlarge), его часто используют для файнтюнинга небольших моделей (7B-14B параметров). В среднем для файнтюнинга достаточно `8-24` часов, поэтому будем предсказывать период в течение суток. Опишу подробнее правила "выживания" SPOT-инстанса в AWS:

1. Выставляется порог max_price, который мы готовы платить, но платим мы по рыночной цене.
2. Если рыночная цена на инстанс становится порога, то инстанс прекращает свою работу и удаляет эфемерное хранилище, что приводит к потере данных.

Следовательно задача такова: предсказание выживания spot-инстанса `g5.2xlarge` в течение суток при предложенном пользователем пороге `max_price` в самом большом регионе `us-east-1` (самый большой регион AWS) c OS `Linux`.

## Добыча данных

Основной данных будет датасет [AWS Spot Price History](https://zenodo.org/records/18821638). В датасете представлены данные с 2022 года по 2026. Они разделены по различным файлам. Скачаем каждый (ОСТОРОЖНО!!! ТУТ 4.4 ГБ ДАННЫХ):

In [1]:
import urllib.request
from pathlib import Path

base_url = "https://zenodo.org/records/18821638/files"
output_dir = Path("../data/")
output_dir.mkdir(parents=True, exist_ok=True)

files = [
    "2022.tsv.zst",
    "2023.tsv.zst",
    "2024-01.tsv.zst", "2024-02.tsv.zst", "2024-03.tsv.zst", "2024-04.tsv.zst",
    "2024-05.tsv.zst", "2024-06.tsv.zst", "2024-07.tsv.zst", "2024-08.tsv.zst",
    "2024-09.tsv.zst", "2024-10.tsv.zst", "2024-11.tsv.zst", "2024-12.tsv.zst",
    "2025-01.tsv.zst", "2025-02.tsv.zst", "2025-03.tsv.zst", "2025-04.tsv.zst",
    "2025-05.tsv.zst", "2025-06.tsv.zst", "2025-07.tsv.zst", "2025-08.tsv.zst",
    "2025-09.tsv.zst", "2025-10.tsv.zst", "2025-11.tsv.zst", "2025-12.tsv.zst",
    "2026-01.tsv.zst", "2026-02.tsv.zst",
]


for filename in files:
    dest = output_dir / filename
    if dest.exists():
        print(f"already exists, skipping: {filename}")
        continue

    url = f"{base_url}/{filename}?download=1"
    print(f"downloading {filename}...")
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"  done: {dest.stat().st_size / 1024 / 1024:.1f} MB")
    except Exception as e:
        print(f"  error: {e}")

print("all done")

already exists, skipping: 2022.tsv.zst
already exists, skipping: 2023.tsv.zst
already exists, skipping: 2024-01.tsv.zst
already exists, skipping: 2024-02.tsv.zst
already exists, skipping: 2024-03.tsv.zst
already exists, skipping: 2024-04.tsv.zst
already exists, skipping: 2024-05.tsv.zst
already exists, skipping: 2024-06.tsv.zst
already exists, skipping: 2024-07.tsv.zst
already exists, skipping: 2024-08.tsv.zst
already exists, skipping: 2024-09.tsv.zst
already exists, skipping: 2024-10.tsv.zst
already exists, skipping: 2024-11.tsv.zst
already exists, skipping: 2024-12.tsv.zst
already exists, skipping: 2025-01.tsv.zst
already exists, skipping: 2025-02.tsv.zst
already exists, skipping: 2025-03.tsv.zst
already exists, skipping: 2025-04.tsv.zst
already exists, skipping: 2025-05.tsv.zst
already exists, skipping: 2025-06.tsv.zst
already exists, skipping: 2025-07.tsv.zst
already exists, skipping: 2025-08.tsv.zst
already exists, skipping: 2025-09.tsv.zst
already exists, skipping: 2025-10.tsv.zs

## Отсортируем данные и объединим датасет

In [2]:
import polars as pl

In [3]:
column_mapping = {
    "column_1": "zone",
    "column_2": "instance",
    "column_3": "os_type",
    "column_4": "cost",
    "column_5": "datetime"
}


final_df = pl.read_csv(
    "../data/2022.tsv.zst",
    separator="\t",
    has_header=False).rename(column_mapping)

final_df.head()

zone,instance,os_type,cost,datetime
str,str,str,f64,str
"""euw3-az2""","""g4dn.4xlarge""","""Linux/UNIX""",0.4664,"""2022-05-31T18:20:41+00:00"""
"""euw3-az2""","""g4dn.4xlarge""","""Red Hat Enterprise Linux""",0.5964,"""2022-05-31T18:20:41+00:00"""
"""euw3-az2""","""g4dn.4xlarge""","""SUSE Linux""",0.5914,"""2022-05-31T18:20:41+00:00"""
"""euw3-az1""","""inf1.xlarge""","""Linux/UNIX""",0.0801,"""2022-05-31T18:21:59+00:00"""
"""euw3-az1""","""inf1.xlarge""","""Red Hat Enterprise Linux""",0.1401,"""2022-05-31T18:21:59+00:00"""


In [4]:
# use1-az* == us-east-1
predicate = (pl.col("instance") == "g5.2xlarge") & (pl.col("os_type") == "Linux/UNIX")

final_df = final_df.filter(predicate)

In [5]:
for filename in files:
    if filename != "2022.tsv.zst":
        dest = output_dir / filename
        temp_df = pl.read_csv(dest, separator="\t", has_header=False).rename(column_mapping)
        temp_df = temp_df.filter(predicate)
        final_df = pl.concat([final_df, temp_df])

In [6]:
final_df.write_csv(output_dir / "g5.2xlarge_in_aws_spot_instances.csv")

In [7]:
final_df.shape

(147210, 5)

In [8]:
final_df

zone,instance,os_type,cost,datetime
str,str,str,f64,str
"""use1-az6""","""g5.2xlarge""","""Linux/UNIX""",0.3636,"""2022-05-31T19:31:56+00:00"""
"""use1-az1""","""g5.2xlarge""","""Linux/UNIX""",0.3636,"""2022-05-31T19:31:56+00:00"""
"""use1-az2""","""g5.2xlarge""","""Linux/UNIX""",0.3636,"""2022-05-31T19:31:56+00:00"""
"""use1-az4""","""g5.2xlarge""","""Linux/UNIX""",0.3636,"""2022-05-31T19:31:56+00:00"""
"""use1-az5""","""g5.2xlarge""","""Linux/UNIX""",0.3636,"""2022-05-31T19:31:56+00:00"""
…,…,…,…,…
"""euw1-az2""","""g5.2xlarge""","""Linux/UNIX""",0.5964,"""2026-02-28T22:46:56Z"""
"""sae1-az1""","""g5.2xlarge""","""Linux/UNIX""",0.879,"""2026-02-28T23:01:06Z"""
"""euw2-az2""","""g5.2xlarge""","""Linux/UNIX""",0.6593,"""2026-02-28T23:16:24Z"""


Данный датасет мы будем изучать в дальнейшем в следующем ноутбуке